# Credit Risk Prediction: Analysis Notebook

**Problem:** a lender must decide whether to approve a loan. Approving someone who later defaults loses the loan amount; rejecting someone who would have repaid loses the interest.
**Task:** predict the probability that an applicant defaults (`loan_status` = 1) and turn it into an approve/reject decision.
**Data:** Kaggle `credit_risk_dataset.csv` (about 32.5k loans, 12 columns).
**Workflow:** load -> EDA -> clean -> split -> pipelines -> compare models (CV) -> handle imbalance -> tune -> threshold -> calibrate -> cost analysis -> SHAP -> error analysis -> save.

## 1. Load the data
We bring in the loan applicant data.
It has details like income, age and loan amount, and it tells us who repaid (0) and who defaulted (1).

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
N_ITER = 150          # RandomizedSearchCV iterations (lower this for a quick run)

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
print("pandas", pd.__version__)

In [ ]:
# Relative path: this notebook lives in notebooks/, the data in data/raw/
DATA_PATH = Path("../data/raw/credit_risk_dataset.csv")
df = pd.read_csv(DATA_PATH)
df.head()

## 2. Exploratory Data Analysis (EDA)
Here we look at the data before touching it.
We want to understand it first, then work on it.

In [ ]:
print(df.shape)
df.info()

### Missing values
This tells us where we may need to fix things later.

In [ ]:
missing = df.isnull().sum()
missing[missing > 0]

### Target balance
How many applicants defaulted vs repaid?
This tells us if the data is balanced or imbalanced.

In [ ]:
print(df["loan_status"].value_counts())
print()
print(df["loan_status"].value_counts(normalize=True).round(3))

sns.countplot(x="loan_status", data=df)
plt.title("Target distribution (0 = repaid, 1 = defaulted)")
plt.xlabel("loan_status")
plt.ylabel("Number of applicants")
plt.show()

### Outlier check
Look for values that seem impossible. These are signs of bad data entries.

In [ ]:
numerical_cols_eda = df.select_dtypes(include=np.number).columns.drop("loan_status")

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, col in zip(axes.ravel(), numerical_cols_eda):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Explicit counts of impossible / suspicious values
print("Age > 100                :", (df["person_age"] > 100).sum())
print("Employment length > 60   :", (df["person_emp_length"] > 60).sum())
print("Employment length > age  :", (df["person_emp_length"] > df["person_age"]).sum())
df.describe()

### Correlation check
See how strongly the numeric features are related to each other.
This helps us spot repeated or linked information.

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

**Your interpretation:** _(after running this cell, write 2-3 sentences in your own words: which features are most related to default? Do loan_int_rate and loan_grade move together? Is loan_percent_income just loan_amnt / person_income?)_

## 3. Data validation and outlier handling
We remove duplicate rows.
We remove rows with an impossible age or work experience.
We remove rows with zero or negative income and loan amount.
We check that the interest rate looks realistic.
We flag (but keep) rows where `loan_percent_income` does not match loan amount / income.

**Important:** rows with a *missing* `person_emp_length` are kept here. They are imputed later inside the pipeline. (A plain `emp_length <= age` filter would silently drop them, because comparisons with NaN are False.)

In [ ]:
df_clean = df.copy()
print(f"Original rows: {len(df_clean)}")

steps = [
    ("duplicate rows",          lambda d: d.drop_duplicates()),
    ("age outside 18-100",      lambda d: d[d["person_age"].between(18, 100)]),
    ("employment length > 60",  lambda d: d[~(d["person_emp_length"] > 60)]),
    ("employment length > age", lambda d: d[~(d["person_emp_length"] > d["person_age"])]),
    ("income <= 0",             lambda d: d[d["person_income"] > 0]),
    ("loan amount <= 0",        lambda d: d[d["loan_amnt"] > 0]),
]
for name, step in steps:
    before = len(df_clean)
    df_clean = step(df_clean)
    print(f"Removed {before - len(df_clean):>5} rows: {name}")

print(f"Rows after cleaning: {len(df_clean)}")
print(f"Missing emp_length kept (to impute): {df_clean['person_emp_length'].isna().sum()}")
print(f"Missing loan_int_rate kept (to impute): {df_clean['loan_int_rate'].isna().sum()}")
print(f"Loan interest rate range: {df_clean['loan_int_rate'].min():.2f} - {df_clean['loan_int_rate'].max():.2f}")

In [ ]:
# Flag rows where loan_percent_income does not match loan_amnt / person_income (kept, not dropped)
implied = df_clean["loan_amnt"] / df_clean["person_income"]
mismatch = (implied - df_clean["loan_percent_income"]).abs() > 0.02
print(f"Rows where loan_percent_income disagrees with loan_amnt/person_income: {mismatch.sum()}")

# Is a missing interest rate informative? (justifies adding a 'missing' indicator later)
print()
print("Default rate by whether loan_int_rate is missing:")
print(df_clean.groupby(df_clean["loan_int_rate"].isna())["loan_status"].mean().round(3))

## 4. Features, target and train-test split
We separate the inputs from the target (did the applicant default?).
The model learns from the training part only. The test part is kept for the final check.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean.drop("loan_status", axis=1)
y = df_clean["loan_status"]      # 1 = defaulted, 0 = repaid

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Default rate  train: %.3f  test: %.3f" % (y_train.mean(), y_test.mean()))

numeric_cols = ["person_age", "person_income", "person_emp_length", "loan_amnt",
                "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length"]
categorical_cols = ["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"]

## 5. Class imbalance
Only about 1 in 5 applicants defaults. If we ignore this, a model can look accurate by just predicting "repaid" for everyone.
Our first tool is **class weights**: we tell the model that a missed defaulter matters more.
`scale_pos_weight = (number of repaid) / (number of defaulted)`.
Later we also compare this against SMOTE.

In [ ]:
neg, pos = np.bincount(y_train)       # neg = repaid (0), pos = defaulted (1)
scale_pos_weight = neg / pos
print(f"repaid: {neg}, defaulted: {pos}, scale_pos_weight = {scale_pos_weight:.2f}")

## 6. Preprocessing: pipelines and ColumnTransformer
Raw data is not ready for a model directly. Numeric and text columns need different steps.
All preprocessing lives **inside** the pipeline, so during cross-validation it is fitted on training folds only (no leakage).

- **Numeric:** fill missing values with the median and add a "was missing" indicator column (a missing interest rate may itself carry signal). Logistic regression also needs scaling; tree models do not.
- **Categorical:** fill missing with "Missing", then one-hot encode.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def make_preprocessor(scale_numeric: bool) -> ColumnTransformer:
    num_steps = [("imputer", SimpleImputer(strategy="median", add_indicator=True))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
    ])

preprocessor_lr = make_preprocessor(scale_numeric=True)    # logistic regression: scale
preprocessor_tree = make_preprocessor(scale_numeric=False) # random forest / XGBoost: no scaling

## 7. Evaluation helper
One common function to check any model on held-out data.
It reports threshold-free metrics (ROC-AUC, PR-AUC) and threshold-dependent ones (precision, recall, F1) and returns them as a dict so we can build comparison tables.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, precision_recall_curve, brier_score_loss)

def evaluate_model(name, model, X_eval, y_eval, threshold=0.5, plots=True, verbose=True):
    proba = model.predict_proba(X_eval)[:, 1]
    pred = (proba >= threshold).astype(int)

    metrics = {
        "model": name,
        "threshold": round(float(threshold), 3),
        "roc_auc": roc_auc_score(y_eval, proba),          # ranking quality, threshold-free
        "pr_auc": average_precision_score(y_eval, proba),  # better than ROC-AUC when classes are imbalanced
        "accuracy": accuracy_score(y_eval, pred),
        "precision": precision_score(y_eval, pred, zero_division=0),  # when we say "default", how often are we right?
        "recall": recall_score(y_eval, pred, zero_division=0),        # of all real defaulters, how many did we catch?
        "f1": f1_score(y_eval, pred, zero_division=0),
    }

    if verbose:
        print(f"{name} (threshold = {threshold:.2f})")
        for k in ["roc_auc", "pr_auc", "accuracy", "precision", "recall", "f1"]:
            print(f"  {k:<10}: {metrics[k]:.3f}")
        print()
        print(classification_report(y_eval, pred, zero_division=0))

    if plots:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        sns.heatmap(confusion_matrix(y_eval, pred), annot=True, fmt="d", cmap="Blues", ax=axes[0])
        axes[0].set_title(f"{name}: confusion matrix")
        axes[0].set_xlabel("Predicted")
        axes[0].set_ylabel("Actual")

        prec, rec, _ = precision_recall_curve(y_eval, proba)
        axes[1].plot(rec, prec)
        axes[1].set_title(f"{name}: precision-recall curve")
        axes[1].set_xlabel("Recall")
        axes[1].set_ylabel("Precision")
        plt.tight_layout()
        plt.show()

    return metrics

## 8. Model comparison with stratified cross-validation
One train-test split can be lucky or unlucky, so we use 5 stratified folds on the training data.
Each fold keeps the same default / repaid ratio.
We compare three model families:

- **Logistic regression:** simple, interpretable baseline.
- **Random forest:** non-linear, little tuning needed.
- **XGBoost:** gradient boosting, usually strongest on tabular data.

We skip accuracy on purpose: with 78% repaid, it is misleading here.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision",
           "precision": "precision", "recall": "recall", "f1": "f1"}

def make_xgb(spw=scale_pos_weight, **kwargs):
    params = dict(n_estimators=300, max_depth=5, learning_rate=0.1,
                  eval_metric="logloss", random_state=RANDOM_STATE)
    params.update(kwargs)
    return XGBClassifier(scale_pos_weight=spw, **params)

models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor_lr),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor_tree),
        ("classifier", RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                              class_weight="balanced_subsample",
                                              random_state=RANDOM_STATE, n_jobs=1)),
    ]),
    "XGBoost": Pipeline([
        ("preprocessor", preprocessor_tree),
        ("classifier", make_xgb()),
    ]),
}

rows = []
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    row = {"model": name}
    for metric in scoring:
        scores = res[f"test_{metric}"]
        row[metric] = f"{scores.mean():.3f} ± {scores.std():.3f}"
    rows.append(row)

cv_table = pd.DataFrame(rows).set_index("model")
cv_table

**Your interpretation:** _(after running this cell, write 2-3 sentences in your own words: which model wins on ROC-AUC and PR-AUC, and by how much? Is the gap bigger than the ± spread across folds? Is the simple model close enough to prefer?)_

## 9. Comparing imbalance strategies
We compare three ways of dealing with imbalance on the same XGBoost model, using CV on the training data only:

1. **None:** train as-is.
2. **Class weights:** `scale_pos_weight` (our default so far).
3. **SMOTE:** create synthetic minority (defaulter) rows. It must run **inside** the CV pipeline so synthetic rows never leak into validation folds.

Read ROC-AUC / PR-AUC first. Precision and recall here are at a fixed 0.5 cutoff, so they mostly reflect how each strategy shifts the probabilities, which the threshold step (section 11) handles properly.

In [ ]:
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("imbalanced-learn not installed: run  pip install imbalanced-learn  to include SMOTE")

strategies = {
    "No handling": Pipeline([("preprocessor", preprocessor_tree), ("classifier", make_xgb(spw=1))]),
    "Class weights": Pipeline([("preprocessor", preprocessor_tree), ("classifier", make_xgb())]),
}
if HAS_IMBLEARN:
    strategies["SMOTE"] = ImbPipeline([
        ("preprocessor", preprocessor_tree),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", make_xgb(spw=1)),
    ])

rows = []
for name, pipe in strategies.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    row = {"strategy": name}
    for metric in scoring:
        row[metric] = f"{res[f'test_{metric}'].mean():.3f}"
    rows.append(row)

pd.DataFrame(rows).set_index("strategy")

**Your interpretation:** _(after running this cell, write 2-3 sentences in your own words: did SMOTE beat class weights on PR-AUC? If not, say why you keep class weights (simpler, no synthetic rows, same or better ranking).)_

## 10. XGBoost hyperparameter tuning
A model has settings called hyperparameters. We search for a good combination using randomized search with cross-validation on the training data.
We optimize **PR-AUC** (`average_precision`) because it focuses on how well we rank the rare class.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

xg_model = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("classifier", make_xgb()),
])

param_dist = {
    "classifier__n_estimators": randint(150, 600),
    "classifier__max_depth": randint(3, 9),
    "classifier__learning_rate": uniform(0.01, 0.5),
    "classifier__subsample": uniform(0.6, 0.4),          # row sampling
    "classifier__colsample_bytree": uniform(0.6, 0.4),   # column sampling
    "classifier__min_child_weight": randint(1, 10),
    "classifier__gamma": uniform(0, 5),
}
# gamma: low -> more splits -> more complex trees -> higher overfitting risk; high -> simpler trees

random_search = RandomizedSearchCV(
    xg_model, param_dist, n_iter=N_ITER, cv=cv,
    scoring="average_precision", n_jobs=-1, random_state=RANDOM_STATE, verbose=1,
)
random_search.fit(X_train, y_train)

best_model = random_search.best_estimator_     # already refit on the full training set
print(f"Best CV PR-AUC after tuning: {random_search.best_score_:.3f}")
random_search.best_params_

## 11. Side-by-side comparison on the held-out test set
Now we fit the other two models on the full training set and look at all three on the test set, once, at the default 0.5 cutoff.

In [ ]:
lr_model = models["Logistic Regression"].fit(X_train, y_train)
rf_model = models["Random Forest"].fit(X_train, y_train)

results = [
    evaluate_model("Logistic Regression", lr_model, X_test, y_test, plots=False, verbose=False),
    evaluate_model("Random Forest",       rf_model, X_test, y_test, plots=False, verbose=False),
    evaluate_model("XGBoost (tuned)",     best_model, X_test, y_test, plots=False, verbose=False),
]
test_table = pd.DataFrame(results).set_index("model").drop(columns="threshold").round(3)
test_table

In [ ]:
_ = evaluate_model("XGBoost (tuned)", best_model, X_test, y_test)

**Your interpretation:** _(after running this cell, write 2-3 sentences in your own words: how much did tuning and the stronger model improve on the logistic baseline? Is test performance close to the CV numbers (a sign of no overfitting)?)_

## 12. Choosing the classification threshold
The model outputs a probability, not a yes/no. We choose a cutoff to turn it into a decision.
We pick the cutoff using **out-of-fold predictions on the training data**, so the test set stays untouched for the final check.

In [ ]:
from sklearn.model_selection import cross_val_predict

oof_prob = cross_val_predict(best_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_train, oof_prob)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)

best_idx = np.argmax(f1_scores[:-1])
best_f1_threshold = thresholds[best_idx]
print(f"F1-optimal threshold (from CV on train): {best_f1_threshold:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(thresholds, precisions[:-1], label="Precision")
plt.plot(thresholds, recalls[:-1], label="Recall")
plt.plot(thresholds, f1_scores[:-1], label="F1")
plt.axvline(best_f1_threshold, color="k", linestyle="--", label="Best F1 threshold")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision / recall / F1 vs threshold (out-of-fold, train)")
plt.legend()
plt.show()

_ = evaluate_model("XGBoost (tuned, F1 threshold)", best_model, X_test, y_test, threshold=best_f1_threshold)

**Your interpretation:** _(after running this cell, write 2-3 sentences in your own words: is the best threshold above or below 0.5, and why? What happens to recall when you raise it?)_

## 13. Probability calibration
A model's probability should mean what it says: of all applicants given 30% risk, about 30% should default.
Because we trained with class weights, the raw probabilities are pushed upward, so they are usually **over-estimates** of the real risk.
We check with a calibration curve and the **Brier score** (lower is better), and try two fixes:

1. **Sigmoid (Platt scaling):** fits a smooth S-curve from old to new probabilities.
2. **Isotonic regression:** a flexible, non-parametric mapping (needs more data).

The ranking quality (ROC-AUC) should barely change; the probabilities become honest.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

cal_sigmoid = CalibratedClassifierCV(best_model, method="sigmoid", cv=5).fit(X_train, y_train)
cal_isotonic = CalibratedClassifierCV(best_model, method="isotonic", cv=5).fit(X_train, y_train)

test_proba = {
    "Uncalibrated": best_model.predict_proba(X_test)[:, 1],
    "Sigmoid": cal_sigmoid.predict_proba(X_test)[:, 1],
    "Isotonic": cal_isotonic.predict_proba(X_test)[:, 1],
}

calib_table = pd.DataFrame({
    name: {"Brier score": brier_score_loss(y_test, p),
           "ROC-AUC": roc_auc_score(y_test, p),
           "Mean predicted risk": p.mean()}
    for name, p in test_proba.items()
}).T.round(4)
print(f"Actual default rate in test set: {y_test.mean():.4f}")
calib_table

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
for name, p in test_proba.items():
    frac_pos, mean_pred = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
    plt.plot(mean_pred, frac_pos, "o-", label=name)
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed default rate")
plt.title("Calibration curve (test set)")
plt.legend()
plt.show()

# Keep whichever calibrated model has the lower Brier score
cal_choice = "Sigmoid" if calib_table.loc["Sigmoid", "Brier score"] <= calib_table.loc["Isotonic", "Brier score"] else "Isotonic"
calibrated_model = cal_sigmoid if cal_choice == "Sigmoid" else cal_isotonic
print("Calibrated model used from here on:", cal_choice)

**Your interpretation:** _(after running this cell, write 2-3 sentences in your own words: did the uncalibrated model over-estimate risk (curve below the diagonal)? Which calibration method gave the lower Brier score, and did ROC-AUC stay about the same?)_

## 14. Business-cost view: false negatives vs false positives
Not all mistakes cost the same.

- **False negative** (approve someone who defaults): we lose a large part of the loan.
- **False positive** (reject someone who would repay): we lose the interest we would have earned.

We **assume** a false negative costs 5x a false positive. These numbers are an illustration; in a real lender you would derive them from loan size, loss given default and interest margin.
Then we pick the threshold that minimizes the average cost per applicant.

With **well-calibrated** probabilities there is also a closed-form answer: reject when `p > cost_FP / (cost_FP + cost_FN)`.

In [ ]:
COST_FN = 5   # ASSUMPTION: missing a defaulter costs 5 units
COST_FP = 1   # ASSUMPTION: rejecting a good customer costs 1 unit

def avg_cost(y_true, proba, thr):
    pred = proba >= thr
    y_arr = np.asarray(y_true)
    fn = ((~pred) & (y_arr == 1)).sum()
    fp = (pred & (y_arr == 0)).sum()
    return (COST_FN * fn + COST_FP * fp) / len(y_arr)

grid = np.linspace(0.01, 0.99, 99)
costs = np.array([avg_cost(y_train, oof_prob, t) for t in grid])
cost_threshold = grid[np.argmin(costs)]
theory_threshold = COST_FP / (COST_FP + COST_FN)

plt.figure(figsize=(7, 4))
plt.plot(grid, costs)
plt.axvline(cost_threshold, color="k", linestyle="--", label=f"Cost-optimal = {cost_threshold:.2f}")
plt.axvline(best_f1_threshold, color="tab:orange", linestyle=":", label=f"F1-optimal = {best_f1_threshold:.2f}")
plt.xlabel("Threshold")
plt.ylabel("Average cost per applicant")
plt.title(f"Cost vs threshold (cost FN = {COST_FN}, cost FP = {COST_FP}; out-of-fold, train)")
plt.legend()
plt.show()
print(f"Theoretical threshold for calibrated probabilities: {theory_threshold:.3f}")

In [ ]:
rows = []
options = [
    ("Default 0.5 (raw)",                         test_proba["Uncalibrated"], 0.5),
    ("F1-optimal threshold (raw)",                test_proba["Uncalibrated"], best_f1_threshold),
    ("Cost-optimal threshold (raw, from OOF)",    test_proba["Uncalibrated"], cost_threshold),
    (f"Theoretical {theory_threshold:.2f} (calibrated)", test_proba[cal_choice], theory_threshold),
]
for label, proba, thr in options:
    pred = proba >= thr
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rows.append({"strategy": label, "threshold": round(float(thr), 3), "FP": fp, "FN": fn,
                 "precision": precision_score(y_test, pred, zero_division=0),
                 "recall": recall_score(y_test, pred, zero_division=0),
                 "avg cost / applicant": avg_cost(y_test, proba, thr)})

# Reference: approve everyone (never flag a default)
rows.append({"strategy": "Approve everyone (no model)", "threshold": np.nan, "FP": 0, "FN": int(y_test.sum()),
             "precision": np.nan, "recall": 0.0, "avg cost / applicant": avg_cost(y_test, np.zeros(len(y_test)), 0.5)})

cost_table = pd.DataFrame(rows).set_index("strategy").round(3)
cost_table

In [ ]:
# The threshold we carry forward. Change to best_f1_threshold if you prefer the F1 view.
FINAL_THRESHOLD = cost_threshold
print("FINAL_THRESHOLD =", round(float(FINAL_THRESHOLD), 3))

**Your interpretation:** _(after running this cell, write 2-3 sentences in your own words: how much cheaper is the cost-optimal threshold than 0.5, and than approving everyone? What happens to the cost-optimal threshold if false negatives cost 10x instead of 5x? Is this assumption realistic? Also note any fairness concern, e.g. a low-threshold model rejects more good customers.)_

## 15. SHAP interpretation: global and local
SHAP explains **why** the model gives a prediction by splitting it into contributions from each feature.
Values are in log-odds: positive pushes toward default, negative toward repaid.
We explain the tuned XGBoost model (`best_model`).

In [ ]:
import shap

pre = best_model.named_steps["preprocessor"]
clf = best_model.named_steps["classifier"]

X_test_t = pre.transform(X_test)
if hasattr(X_test_t, "toarray"):
    X_test_t = X_test_t.toarray()

# Readable names: drop the "num__" / "cat__" prefixes
feature_names = [n.split("__", 1)[-1] for n in pre.get_feature_names_out()]
X_test_df = pd.DataFrame(X_test_t, columns=feature_names, index=X_test.index)

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test_df)
base_value = float(np.ravel(explainer.expected_value)[0])
print("SHAP values shape:", shap_values.shape)

### Global explanation
Which features matter most overall, and in which direction do they push risk?

In [ ]:
shap.summary_plot(shap_values, X_test_df, show=True)

In [ ]:
shap.summary_plot(shap_values, X_test_df, plot_type="bar", show=True)

mean_abs = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_names).sort_values(ascending=False)
mean_abs.head(10).round(3)

**Your interpretation:** _(after running this cell, write 2-3 sentences in your own words: which 3-5 features matter most? For the top ones, does a high value push risk up or down, and does that make business sense (e.g. high interest rate, high loan-to-income, renting vs owning)?)_

### Local explanation
Why did the model score one specific applicant the way it did?
We look at the highest-risk and the lowest-risk applicant in the test set.

In [ ]:
test_proba_best = test_proba["Uncalibrated"]

def explain_applicant(pos, title):
    print(f"{title}: test row {pos}, model risk = {test_proba_best[pos]:.2f}, actual loan_status = {y_test.iloc[pos]}")
    shap.plots.waterfall(
        shap.Explanation(values=shap_values[pos], base_values=base_value,
                         data=X_test_df.iloc[pos].values, feature_names=feature_names),
        max_display=12,
    )

explain_applicant(int(np.argmax(test_proba_best)), "Highest-risk applicant")

In [ ]:
explain_applicant(int(np.argmin(test_proba_best)), "Lowest-risk applicant")

**Your interpretation:** _(after running this cell, write 2-3 sentences in your own words: what pushed the high-risk applicant up and the low-risk applicant down? How would you explain the high-risk decision to the applicant in one plain sentence?)_

## 16. Error analysis: false positives and false negatives
The model will not be perfect.
**False positives:** safe people wrongly marked risky.
**False negatives:** risky people wrongly marked safe.
We study them at our chosen threshold to see where the model struggles.

In [ ]:
err = X_test.copy()
err["actual"] = y_test
err["proba"] = test_proba_best
err["pred"] = (err["proba"] >= FINAL_THRESHOLD).astype(int)
err["outcome"] = np.select(
    [(err.actual == 1) & (err.pred == 1), (err.actual == 0) & (err.pred == 0),
     (err.actual == 0) & (err.pred == 1), (err.actual == 1) & (err.pred == 0)],
    ["TP", "TN", "FP", "FN"],
    default="",
)
print(err["outcome"].value_counts())
print()
print("Median of numeric features by outcome:")
err.groupby("outcome")[numeric_cols + ["proba"]].median().round(2)

In [ ]:
print("Loan grade mix within each outcome (columns sum to 1):")
pd.crosstab(err["loan_grade"], err["outcome"], normalize="columns").round(2)

In [ ]:
print("Default rate and flagged rate by loan grade:")
by_grade = err.groupby("loan_grade").agg(applicants=("actual", "size"),
                                         actual_default_rate=("actual", "mean"),
                                         flagged_rate=("pred", "mean")).round(2)
by_grade

**Your interpretation:** _(after running this cell, write 2-3 sentences in your own words: how do the false negatives (missed defaulters) differ from correctly caught ones? Are errors concentrated in certain loan grades or income ranges? What data would help the model catch them?)_

## 17. Final model selection and saving
We keep the tuned XGBoost model, its calibrated version for honest probabilities, and the chosen threshold.
The saved files go to `artifacts/`, which `.gitignore` keeps out of version control.

In [ ]:
final = evaluate_model("FINAL: tuned XGBoost", best_model, X_test, y_test,
                       threshold=FINAL_THRESHOLD, plots=False, verbose=False)
pd.Series({k: v for k, v in final.items() if k != "model"}).round(3)

In [ ]:
MODEL_DIR = Path("../artifacts/model")
THRESH_DIR = Path("../artifacts/threshold")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
THRESH_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, MODEL_DIR / "credit_risk_xgb.joblib")
joblib.dump(calibrated_model, MODEL_DIR / "credit_risk_xgb_calibrated.joblib")

config = {
    "decision_threshold": float(FINAL_THRESHOLD),
    "f1_threshold": float(best_f1_threshold),
    "cost_threshold": float(cost_threshold),
    "cost_fn": COST_FN, "cost_fp": COST_FP,
    "calibration_method": cal_choice,
    "best_params": {k: (float(v) if isinstance(v, (float, np.floating)) else int(v))
                    for k, v in random_search.best_params_.items()},
    "test_metrics": {k: (round(float(v), 4) if not isinstance(v, str) else v) for k, v in final.items()},
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
}
(THRESH_DIR / "model_config.json").write_text(json.dumps(config, indent=2))

print("Model files    :", sorted(p.name for p in MODEL_DIR.iterdir()))
print("Threshold files:", sorted(p.name for p in THRESH_DIR.iterdir()))

## Limitations and next steps
- **Dataset:** a public Kaggle dataset with no time dimension, so we cannot test how the model holds up over time.
- **Lender-assigned features:** `loan_grade` and `loan_int_rate` are produced by the lender's own risk assessment. They are strong signals, but in a real system you would ask whether they are available at decision time and whether they hide bias.
- **Costs are assumed:** the 5:1 cost ratio is an illustration, not a measured business number.
- **Fairness:** age and home ownership can act as proxies for protected attributes. A real deployment needs a fairness audit across groups.
- **Next:** move the final pipeline into `train.py`, add a small API / demo app, and write the README with results and SHAP plots.